# Pipeline cancellation

Cancel queued or running work, inspect the acknowledgement, and reopen an exact run.
The examples use the installed `Wait` operation and the observed local worker.
No provider account or GPU is required.

Cancellation requests do not prove that work has stopped. Only confirmed closure
allows a cancelled result; uncertain ownership remains `unknown` and prevents
resume. Cancellation does not undo external side effects.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import Wait
from artisan.orchestration import PipelineManager, StepDisposition, StepStatus
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

In [ ]:
env = tutorial_setup("pipeline_cancellation")

## Programmatic cancellation

Build a 4-step pipeline using `Wait` (a lightweight operation that
sleeps for a configurable duration). Run the first two steps, call
`pipeline.cancel()`, then run the remaining two. Those new attempts
record confirmed cancellation without dispatching execution work.

In [ ]:
pipeline = PipelineManager.create(
    name="cancellation_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

step_0 = pipeline.run(
    definition=Wait,
    name="wait_0",
    params={"duration": 0.1},
)
step_1 = pipeline.run(
    definition=Wait,
    name="wait_1",
    params={"duration": 0.11},
)
print(f"Step 0: {step_0.succeeded_count} succeeded")
print(f"Step 1: {step_1.succeeded_count} succeeded")

Steps 0–1 completed normally. Now cancel the pipeline and run
two more steps:

In [ ]:
pipeline.cancel()

step_2 = pipeline.run(
    definition=Wait,
    name="wait_2",
    params={"duration": 0.12},
)
step_3 = pipeline.run(
    definition=Wait,
    name="wait_3",
    params={"duration": 0.13},
)

result = pipeline.finalize()
print(f"Pipeline: {result['total_steps']} steps, success={result['overall_success']}")
assert step_2.status is StepStatus.CANCELLED
assert step_3.status is StepStatus.CANCELLED
assert not result["overall_success"]

Steps 2–3 became `cancelled` without dispatching an execution. Their
histories contain `pending`, cancellation `requested`, cancellation
`confirmed`, and the terminal `cancelled` snapshot.

## Inspecting cancelled results

Steps 2–3 report `status=StepStatus.CANCELLED` and
`cancellation_status=confirmed`. They never dispatched work, so there is
nothing left to stop.

For active work, confirmation requires evidence of termination. If work
completed before cancellation, the request can be `rejected` and its natural
result remains available. If termination cannot be established, the
acknowledgement is `unknown`: the attempt fails and accepts no outputs.

In [ ]:
for step in [step_0, step_1, step_2, step_3]:
    if step.status is StepStatus.CANCELLED:
        print(
            f"Step {step.step_number} ({step.step_name}): "
            f"cancelled — {step.cancellation_status.value}"
        )
    else:
        print(
            f"Step {step.step_number} ({step.step_name}): "
            f"{step.succeeded_count}/{step.total_count} succeeded"
        )

These cancelled steps expose no outputs and have zero counts. The pipeline
summary reports `overall_success=False`.

In [ ]:
inspect_pipeline(env.delta_root)

`inspect_pipeline` shows the current attempt for each step in the selected
run. Steps 2–3 appear as `cancelled` even though they never dispatched work.
Pass `pipeline_run_id` when the store contains multiple runs; the default
selects the latest run. Inspect `StepResult.cancellation_status` for the
typed acknowledgement.

## Start a new run after cancellation

The default local worker observes the current interpreter and source. Its work
is deliberately not reusable as a qualified scientific cache entry. All four
steps below therefore execute in the new run; the cancelled attempts never
supply results. The [cache tutorial](../03-caching/01-resume-and-caching.ipynb)
prepares a canonical worker and demonstrates reuse separately.

In [ ]:
env = tutorial_setup("pipeline_cancellation", clean=False)

pipeline2 = PipelineManager.create(
    name="cancellation_rerun",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

In [ ]:
step_0r = pipeline2.run(
    definition=Wait,
    name="wait_0",
    params={"duration": 0.1},
)
step_1r = pipeline2.run(
    definition=Wait,
    name="wait_1",
    params={"duration": 0.11},
)
step_2r = pipeline2.run(
    definition=Wait,
    name="wait_2",
    params={"duration": 0.12},
)
step_3r = pipeline2.run(
    definition=Wait,
    name="wait_3",
    params={"duration": 0.13},
)

result2 = pipeline2.finalize()
print(f"Pipeline: {result2['total_steps']} steps, success={result2['overall_success']}")
assert step_0r.disposition is StepDisposition.EXECUTED
assert step_1r.disposition is StepDisposition.EXECUTED
assert step_2r.disposition is StepDisposition.EXECUTED
assert step_3r.disposition is StepDisposition.EXECUTED
assert result2["overall_success"]

Both completed and previously cancelled computations execute under this
observed worker. The original run remains available for inspection.

In [ ]:
for step in [step_0r, step_1r, step_2r, step_3r]:
    print(
        f"Step {step.step_number} ({step.step_name}): "
        f"{step.status.value}, {step.disposition.value}: "
        f"{step.succeeded_count}/{step.total_count} succeeded"
    )

In [ ]:
inspect_pipeline(env.delta_root)

(recover-finished-units)=
## Reopen accepted work

Resume preserves the same run and its exact accepted outputs. It may finish a
previously selected immutable result plan; it does not adopt an unclaimed seal,
retry a failed slot, or rerun science. Pending, running, or unknown ownership
must be resolved before new work can start.

The second run above is already complete. Reopening it is a simple observable
case: its output IDs and protected origins stay unchanged, and a new step is
appended after the existing four.

In [ ]:
from artisan.visualization import inspect_step

run_id = pipeline2.config.pipeline_run_id
before = inspect_step(env.delta_root, 0, pipeline_run_id=run_id)
with PipelineManager.resume(
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    pipeline_run_id=run_id,
) as resumed:
    assert resumed.config.pipeline_run_id == run_id
    appended = resumed.run(Wait, name="after_resume", params={"duration": 0.01})
    resumed_summary = resumed.finalize()
after = inspect_step(env.delta_root, 0, pipeline_run_id=run_id)
assert before.equals(after)
assert appended.step_number == 4
assert resumed_summary["overall_success"]
assert inspect_step(
    env.delta_root, 2, pipeline_run_id=pipeline.config.pipeline_run_id
).is_empty()

Use `preserve_staging=True` when you need successful staging evidence for
inspection. Unselected evidence is never permission to publish results. Read-only
store inspection reports incomplete selections and orphan evidence separately;
selected-plan recovery preserves the original scientific decision.

## Signals and active work

The first Ctrl+C or SIGTERM requests cancellation through the active owner. A
second signal restores previous handlers. Further signals follow those handlers.
The acknowledgement describes what the owner could verify, not just whether a
signal was sent.

Run the companion script from a terminal to exercise an active worker:

```bash
pixi run python docs/tutorials/05-errors-and-control/cancel_demo.py
```

It submits ten installed `Wait` steps. Press Ctrl+C while a step runs, then inspect
the succeeded and cancelled results. In a notebook, call `pipeline.cancel()`
directly instead of signalling the notebook kernel.

Managed commands and provider workers have their own closure evidence. An SDK
cancellation acknowledgement or a stopped launcher alone is insufficient.
Uncertain cleanup retains the affected workspace and blocks output collection.
See [Configure Execution](../../how-to-guides/configuring-execution.md) and
[Run Remote Workers](../../how-to-guides/running-remote-workers.md).

## Next steps

- [Resume and caching](../03-caching/01-resume-and-caching.ipynb)
- [Error visibility](02-error-visibility.ipynb)
- [Storage and recovery](../../concepts/storage-and-delta-lake.md)